In [ ]:
import pandas as pd # librería principal para trabajar con tablas
from sklearn.preprocessing import MinMaxScaler # herramienta para escalar valores entre 0 y 1
import unicodedata # librería estándar, sirve para manejar tildes

In [ ]:
# 1.Exploracion iniciarl 
df = pd.read_csv("Data/datos_para_limpiar.csv") # La ruta es relativa: el archivo debe estar en la carpeta "Data" junto al script.

In [ ]:
print("Primeras 5 filas:")# head() muestra las primeras 5 filas: sirve para ver cómo lucen los datos.
print(df.head())

Primeras 5 filas:
   ID  Nombre  Edad       Salario  Ciudad      Genero   Nivel       Fecha  \
0   1     Ana   NaN  3.955413e+03  bogota   Masculino    Bajo  2018-12-29   
1   2   Maria   NaN  3.054891e+03     NaN    Femenino    alto  2011-07-16   
2   3  Carlos   NaN  2.928345e+08  bogota    Femenino    Bajo  2022-04-07   
3   4   Maria  11.0  4.718402e+03     NaN         NaN  medio   2022-10-11   
4   5   Laura  94.0  7.477876e+07    cali  masculino     alto  2017-07-11   

      Ingresos       Gastos    Codigo                 Notas  
0  4030.026008  1213.936751  ABC12345  Pendiente evaluación  
1  7509.810584  6038.896355       NaN         Buen empleado  
2  9106.362328  6705.724812       NaN                   NaN  
3          NaN          NaN  XYZ98765                   NaN  
4  3102.167244  7570.107277       NaN  Pendiente evaluación  


In [ ]:
print("Información general:") # info() muestra el tipo de dato de cada columna y cuántos valores NO nulos hay.
# Aquí se ve, por ejemplo, que "Fecha" es texto (str) y debería ser fecha.
df.info()

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        100 non-null    int64  
 1   Nombre    100 non-null    str    
 2   Edad      50 non-null     float64
 3   Salario   65 non-null     float64
 4   Ciudad    82 non-null     str    
 5   Genero    73 non-null     str    
 6   Nivel     73 non-null     str    
 7   Fecha     100 non-null    str    
 8   Ingresos  48 non-null     float64
 9   Gastos    53 non-null     float64
 10  Codigo    69 non-null     str    
 11  Notas     67 non-null     str    
dtypes: float64(4), int64(1), str(7)
memory usage: 9.5 KB


In [ ]:
print("Nulos por columna:") # isnull() marca con True cada celda vacía y sum() las cuenta por columna.
print(df.isnull().sum())

Nulos por columna:
ID           0
Nombre       0
Edad        50
Salario     35
Ciudad      18
Genero      27
Nivel       27
Fecha        0
Ingresos    52
Gastos      47
Codigo      31
Notas       33
dtype: int64


In [ ]:
# 2. ELIMINAR COLUMNAS INNECESARIAS (ID y Notas)
df = df.drop(columns=["ID", "Notas"]) # columns=[...] le dice a drop() que son columnas (no filas) lo que quitamos.

In [ ]:
#3. NORMALIZAR NOMBRES DE CoLUMNAS
#minúsculas, sin espacios y sin tildes
#   1. normalize("NFKD") separa cada letra de su tilde: "í" pasa a "i" + "´"
#   2. combining(c) detecta cuáles caracteres son solo tildes
#   3. nos quedamos únicamente con los que NO son tildes
def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

df.columns = [
    quitar_tildes(c.strip().lower()).replace(" ", "_") for c in df.columns
]


In [10]:
# 4. CORREGIR ERRORES DE ESCRITURA EN CIUDAD
#    Se quitan espacios, mayúsculas y tildes para comparar,
#    y luego se asigna el nombre oficial de cada ciudad.
mapa_ciudades = {"bogota": "Bogotá", "medellin": "Medellín", "cali": "Cali"}
 
df["ciudad"] = (
    df["ciudad"]
    .str.strip()
    .str.lower()
    .apply(lambda x: quitar_tildes(x) if isinstance(x, str) else x)
    .map(mapa_ciudades)          # los nulos siguen siendo nulos
)

In [ ]:
# 5. TRATAMIENTO DE VALORES NULOS
# edad: media calculada solo con edades válidas (0 a 120)
# Hay edades imposibles (-3, 125, 129). Si las incluyéramos en la media, la
# dañarían. Por eso primero filtramos las edades entre 0 y 120 y calculamos
# la media solo con esas.
edades_validas = df.loc[df["edad"].between(0, 120), "edad"]
df["edad"] = df["edad"].fillna(edades_validas.mean())

In [ ]:
# salario: mediana
# La mediana es mejor que la media cuando hay valores extremos, y el salario
# tiene muchos (hay valores de cientos de millones).
df["salario"] = df["salario"].fillna(df["salario"].median())

In [ ]:
# ciudad y genero: moda
# La moda es el valor que más se repite. Sirve para texto, donde no se puede
# calcular promedio. mode() devuelve una lista (puede haber empates) y [0]
# toma el primero.
df["ciudad"] = df["ciudad"].fillna(df["ciudad"].mode()[0])
df["genero"] = df["genero"].fillna(df["genero"].mode()[0])

In [ ]:
#6. ASEGURAR TIPOS DE DATOS
# to_numeric convierte a número. errors="coerce" significa: si algún valor no se
# puede convertir, no te sale error, lo vuelve nulo (NaN).
df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

In [ ]:
# 7. FILTRAR DATOS FUERA DE RANGO
# Filtrar en pandas = quedarnos con las filas que cumplen una condición.
# Edades entre 0 y 120: se descartan -3, 125, 129, etc
df = df[df["edad"].between(0, 120)]          # edades entre 0 y 120
df = df[df["salario"] <= 1_000_000_000]      # salarios hasta 1.000 millones

In [ ]:
#8. VARIABLE ORDINAL: nivel -> número
# map() reemplaza cada valor usando el diccionario. Lo que NO esté en el
# diccionario (vacíos o "n/a") se vuelve NaN automáticamente, que es justo lo
# que pide el taller: mantenerlos como nulos.
df["nivel"] = df["nivel"].map({"bajo": 1, "medio": 2, "alto": 3})

In [ ]:
# 9. CODIFICACIÓN DE VARIABLES CATEGÓRICAS (one-hot)
# drop_first=True elimina la primera categoría de cada variable. Sirve para evitar
# la multicolinealidad: si sabes que no es Cali ni Medellín, ya sabes que es
# Bogotá, o sea, esa columna sobra.
# dtype=int deja las columnas en 0 y 1 (por defecto salen como True/False).
# Esto reemplaza las columnas "genero" y "ciudad" originales por las nuevas.
# Por eso después de este paso df["ciudad"] ya NO existe.
df = pd.get_dummies(df, columns=["genero", "ciudad"], drop_first=True, dtype=int)

In [ ]:
# 10. ESCALADO DE edad E ingresos (0 a 1)
# MinMaxScaler ignora los NaN al calcular min y max
# edad va de 0 a 120 e ingresos de miles, o sea, escalas muy distintas. Muchos
# algoritmos se confunden con eso. MinMaxScaler lleva todo al rango 0 a 1 con:
#   valor_nuevo = (valor - mínimo) / (máximo - mínimo)
scaler = MinMaxScaler()
df[["edad", "ingresos"]] = scaler.fit_transform(df[["edad", "ingresos"]])

In [ ]:
# 11. ELIMINAR DUPLICADOS
# drop_duplicates() borra las filas que son idénticas en TODAS las columnas,
# dejando la primera que aparece.
df = df.drop_duplicates()


In [ ]:
# 12. NUEVA VARIABLE: utilidad = ingresos - gastos
#en el paso 10 "ingresos" quedó escalado entre 0 y 1, mientras que
# "gastos" sigue en su valor real. Por eso esta resta mezcla escalas. Se deja en
# este orden porque es el que pide el taller. Si se quiere la utilidad en pesos
# reales, este paso debería hacerse ANTES del paso 10.
df["utilidad"] = df["ingresos"] - df["gastos"]

In [ ]:
# 13. GUARDAR EL DATASET LIMPIO
# index=False evita que pandas agregue una columna extra con el número de fila.
df.to_csv("datos_limpios.csv", index=False)
